## 🎙️ Media Download + Whisper Transcription / Translation Pipeline

**Quick start:**
1. Set runtime to GPU — *Runtime → Change runtime type → T4 GPU*
2. Fill in the form below and click **▶ Run Pipeline**
3. Select *Runtime → Run all* (Ctrl+F9) — the rest runs automatically and the runtime disconnects when done

> Supports YouTube, Vimeo, Twitter/X, Bilibili, Twitch, DoodStream, Streamtape, Voe.sx, Mxdrop, and hundreds more. Translation via Gemini 1.5 Flash (free tier, no credit card) — context-aware, resolves implicit subjects.

In [1]:
# @title CONFIG
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎙️  Whisper Pipeline – Configuration                             ║
# ║  Fill in the options, then  Runtime → Run all  (Ctrl+F9)         ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, subprocess, time, json, textwrap
from pathlib import Path
from datetime import timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML as IPHTML

# ─── Card header ───────────────────────────────────────────────────────────
_style = """
<style>
.pip-sec  { font-weight:600; font-size:.9rem; color:#343a40; margin:14px 0 3px 0 }
.pip-hint { color:#6c757d; font-size:.78rem; margin:1px 0 6px 0 }
</style>
<div style="background:#f0f4ff;border:1px solid #c5d0e6;border-radius:10px;
            padding:16px 20px;max-width:740px;
            font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif">
  <h2 style="margin:0 0 4px 0;font-size:1.2rem;color:#1a1a2e">
      🎙️ Media Download + Whisper Pipeline</h2>
  <p style="color:#6c757d;font-size:.85rem;margin:0">
      Configure below, click <b>▶ Run Pipeline</b>, then
      <b>Runtime → Run all</b> — everything runs automatically.</p>
</div>
"""
display(IPHTML(_style))

# ─── INPUT ─────────────────────────────────────────────────────────────────
display(IPHTML('<div class="pip-sec">📥 Where is your media?</div>'))

input_mode_w = widgets.ToggleButtons(
    options=[
        ("🔗  URLs (yt-dlp)", "urls"),
        ("🗂️  Google Drive",   "drive"),
        ("⬆️  Upload files",   "session"),
    ],
    value="drive",
    button_style="",
)

url_box = widgets.Textarea(
    placeholder="Paste one URL per line…",
    layout=widgets.Layout(width="100%", height="110px"),
)
drive_in_w = widgets.Text(
    value="/content/drive/MyDrive/WhisperJAV",
    placeholder="Google Drive folder path",
    layout=widgets.Layout(width="100%"),
)
urls_panel = widgets.VBox([
    widgets.HTML('<div class="pip-hint">Supports YouTube, Vimeo, Twitter/X, Bilibili, Twitch, and hundreds more.</div>'),
    url_box,
])
drive_in_panel = widgets.VBox([
    widgets.HTML('<div class="pip-hint">Drive is mounted automatically.</div>'),
    drive_in_w,
])
session_in_panel = widgets.HTML(
    '<div class="pip-hint">A file-picker dialog appears automatically in the next step.</div>'
)
input_sub = widgets.VBox([drive_in_panel])

def _on_input(change):
    v = change["new"]
    if v == "urls":
        input_sub.children = [urls_panel]
    elif v == "drive":
        input_sub.children = [drive_in_panel]
    else:
        input_sub.children = [session_in_panel]
input_mode_w.observe(_on_input, names="value")
display(widgets.VBox([input_mode_w, input_sub]))

# ─── OUTPUT ────────────────────────────────────────────────────────────────
display(IPHTML('<hr style="margin:10px 0;border-color:#dee2e6">'))
display(IPHTML('<div class="pip-sec">💾 Where should results go?</div>'))

output_mode_w = widgets.ToggleButtons(
    options=[
        ("⬇️  Download ZIP",  "session"),
        ("🗂️  Google Drive",   "drive"),
    ],
    value="drive",
    button_style="",
)
drive_out_w = widgets.Text(
    value="/content/drive/MyDrive/WhisperJAV/output",
    placeholder="Google Drive folder path",
    layout=widgets.Layout(width="100%"),
)
session_out_panel = widgets.HTML(
    '<div class="pip-hint">A .zip with all transcripts downloads automatically when done.</div>'
)
drive_out_panel = widgets.VBox([
    widgets.HTML('<div class="pip-hint">Folder is created if it does not exist.</div>'),
    drive_out_w,
])
output_sub = widgets.VBox([drive_out_panel])

def _on_output(change):
    output_sub.children = [drive_out_panel if change["new"] == "drive" else session_out_panel]
output_mode_w.observe(_on_output, names="value")
display(widgets.VBox([output_mode_w, output_sub]))

# ─── OUTPUT FORMATS ────────────────────────────────────────────────────────
display(IPHTML('<hr style="margin:10px 0;border-color:#dee2e6">'))
display(IPHTML('<div class="pip-sec">📄 Output formats</div>'))
display(IPHTML('<div class="pip-hint">Choose which files to generate. SRT is always produced.</div>'))

fmt_txt_w  = widgets.Checkbox(value=False, description=".txt  — plain text transcript",    indent=False)
fmt_json_w = widgets.Checkbox(value=False, description=".json — full segment metadata",    indent=False)
fmt_orig_w = widgets.Checkbox(value=False, description=".{lang}.txt — original language text (before translation)", indent=False)

display(widgets.VBox([fmt_txt_w, fmt_json_w, fmt_orig_w]))

# ─── WHISPER SETTINGS ──────────────────────────────────────────────────────
display(IPHTML('<hr style="margin:10px 0;border-color:#dee2e6">'))
display(IPHTML('<div class="pip-sec">🎙️ Whisper settings</div>'))
display(IPHTML('<div class="pip-hint">Larger model = slower but more accurate. Beam size 5 is a good default.</div>'))

model_w = widgets.Dropdown(
    options=["tiny", "base", "small", "medium", "large-v2", "large-v3"],
    value="large-v3",
    description="Model:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="210px"),
)
task_w = widgets.ToggleButtons(
    options=[
        ("📝  Transcribe (keep language)", "transcribe"),
        ("🌐  Translate to English",       "translate"),
    ],
    value="transcribe",
    button_style="",
)
lang_w = widgets.Text(
    value="ja",
    description="Language:",
    placeholder="e.g. ja, en, es, fr  —  blank = auto-detect",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="280px"),
)
beam_w = widgets.IntSlider(
    value=5, min=1, max=10,
    description="Beam size:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="340px"),
)
display(model_w)
display(task_w)
display(widgets.HBox([lang_w, beam_w]))

# ─── TRANSLATION SETTINGS ────────────────────────────────────────────
display(IPHTML('<hr style="margin:10px 0;border-color:#dee2e6">'))
display(IPHTML('<div class="pip-sec">🔑 Translation settings (Gemini API)</div>'))
display(IPHTML('<div class="pip-hint">Only used when task = Translate to English. '
               'Gemini resolves implicit subjects (he/she vs I) from context. '
               'Free key, no credit card: aistudio.google.com</div>'))

gemini_key_w = widgets.Password(
    placeholder="AIza…",
    description="API key:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
display(gemini_key_w)

# ─── RUN BUTTON ────────────────────────────────────────────────────────────
display(IPHTML('<hr style="margin:10px 0;border-color:#dee2e6">'))

run_btn    = widgets.Button(
    description="  Run Pipeline",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(height="42px", width="200px"),
)
status_out = widgets.Output()

def _on_run(_):
    global INPUT_MODE, DOWNLOAD_URLS, DRIVE_INPUT_FOLDER, SESSION_INPUT_FOLDER
    global OUTPUT_MODE, SESSION_OUTPUT_FOLDER, DRIVE_OUTPUT_FOLDER
    global MODEL_NAME, SOURCE_LANGUAGE, TASK, BEAM_SIZE
    global COMPUTE_TYPE_GPU, COMPUTE_TYPE_CPU
    global VIDEO_EXTENSIONS, AUDIO_EXTENSIONS, SUPPORTED_EXTENSIONS
    global SAVE_TXT, SAVE_JSON, SAVE_ORIG_TXT, GEMINI_API_KEY

    INPUT_MODE  = input_mode_w.value
    OUTPUT_MODE = output_mode_w.value

    raw_urls      = url_box.value.strip().splitlines()
    DOWNLOAD_URLS = [u.strip() for u in raw_urls if u.strip()]

    DRIVE_INPUT_FOLDER    = drive_in_w.value.strip()  or "/content/drive/MyDrive/WhisperJAV"
    DRIVE_OUTPUT_FOLDER   = drive_out_w.value.strip() or "/content/drive/MyDrive/WhisperJAV/output"
    SESSION_INPUT_FOLDER  = "/content/input"
    SESSION_OUTPUT_FOLDER = "/content/output"

    MODEL_NAME       = model_w.value
    TASK             = task_w.value
    SOURCE_LANGUAGE  = lang_w.value.strip() or None
    BEAM_SIZE        = beam_w.value
    COMPUTE_TYPE_GPU = "float16"
    COMPUTE_TYPE_CPU = "int8"

    SAVE_TXT         = fmt_txt_w.value
    SAVE_JSON        = fmt_json_w.value
    SAVE_ORIG_TXT    = fmt_orig_w.value
    GEMINI_API_KEY   = gemini_key_w.value.strip()

    VIDEO_EXTENSIONS     = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".ts"}
    AUDIO_EXTENSIONS     = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}
    SUPPORTED_EXTENSIONS = VIDEO_EXTENSIONS | AUDIO_EXTENSIONS

    fmts = ["srt"] + (["txt"] if SAVE_TXT else []) + (["json"] if SAVE_JSON else []) + (["lang.txt"] if SAVE_ORIG_TXT else [])

    with status_out:
        clear_output()
        print("Configuration saved!")
        url_info = f"  ({len(DOWNLOAD_URLS)} URL(s))" if INPUT_MODE == "urls" else ""
        print(f"   Input   : {INPUT_MODE}{url_info}")
        print(f"   Output  : {OUTPUT_MODE}")
        print(f"   Model   : {MODEL_NAME}  |  task={TASK}  |  lang={SOURCE_LANGUAGE or 'auto'}  |  beam={BEAM_SIZE}")
        print(f"   Formats : {', '.join(fmts)}")
        print()
        print("Now select  Runtime > Run all  (or Ctrl+F9)  to start the pipeline.")

run_btn.on_click(_on_run)
display(widgets.VBox([run_btn, status_out]))

# ─── Safe defaults (downstream cells work even if Run is never clicked) ─────
INPUT_MODE            = input_mode_w.value
OUTPUT_MODE           = output_mode_w.value
DOWNLOAD_URLS         = []
DRIVE_INPUT_FOLDER    = drive_in_w.value
SESSION_INPUT_FOLDER  = "/content/input"
SESSION_OUTPUT_FOLDER = "/content/output"
DRIVE_OUTPUT_FOLDER   = drive_out_w.value
MODEL_NAME            = model_w.value
SOURCE_LANGUAGE       = lang_w.value.strip() or None
TASK                  = task_w.value
BEAM_SIZE             = beam_w.value
COMPUTE_TYPE_GPU      = "float16"
COMPUTE_TYPE_CPU      = "int8"
SAVE_TXT              = fmt_txt_w.value
SAVE_JSON             = fmt_json_w.value
SAVE_ORIG_TXT         = fmt_orig_w.value
GEMINI_API_KEY        = gemini_key_w.value.strip()
VIDEO_EXTENSIONS      = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".ts"}
AUDIO_EXTENSIONS      = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg"}
SUPPORTED_EXTENSIONS  = VIDEO_EXTENSIONS | AUDIO_EXTENSIONS


Dropdown(description='Model:', index=5, layout=Layout(width='210px'), options=('tiny', 'base', 'small', 'mediu…

ToggleButtons(options=(('📝  Transcribe (keep language)', 'transcribe'), ('🌐  Translate to English', 'translate…

Password(description='API key:', layout=Layout(width='420px'), placeholder='AIza…', style=DescriptionStyle(des…

In [2]:
# @title ⚙️ Step 1 — Install dependencies
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

print("Installing dependencies...")

!pip -q install -U faster-whisper ctranslate2 pandas tqdm yt-dlp playwright jsbeautifier nest_asyncio requests
!playwright install chromium
!playwright install-deps chromium
!apt-get -qq update
!apt-get -qq install -y ffmpeg

print("Done.")


Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# @title ⚙️ Step 2 — Mount Google Drive & set up folders
# ============================================================
# GOOGLE DRIVE + FOLDER SETUP
# ============================================================

if INPUT_MODE == "drive" or OUTPUT_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')

if OUTPUT_MODE == "drive":
    OUTPUT_FOLDER = DRIVE_OUTPUT_FOLDER
else:
    OUTPUT_FOLDER = SESSION_OUTPUT_FOLDER

if INPUT_MODE == "urls":
    INPUT_FOLDER = OUTPUT_FOLDER  # downloads land alongside their transcripts
elif INPUT_MODE == "drive":
    INPUT_FOLDER = DRIVE_INPUT_FOLDER
else:
    INPUT_FOLDER = SESSION_INPUT_FOLDER

Path(INPUT_FOLDER).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

print(f"Input folder:  {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")

Mounted at /content/drive
Input folder:  /content/drive/MyDrive/WhisperJAV
Output folder: /content/drive/MyDrive/WhisperJAV/output


In [ ]:
# @title ⚙️ Step 3 — Upload files (only when Input = Upload)
# ============================================================
# UPLOAD FILES (only runs if INPUT_MODE == "session")
# ============================================================

if INPUT_MODE == "session":
    from google.colab import files as colab_files

    print("Select the audio/video file(s) to upload:")
    uploaded = colab_files.upload()

    for name in uploaded:
        target = Path(INPUT_FOLDER) / name
        Path(name).rename(target)
        print(f"Saved -> {target}")
else:
    print("Skipping upload step (INPUT_MODE != 'session').")

Skipping upload step (INPUT_MODE != 'session').


In [ ]:
# @title ⚙️ Step 4 — Download media (only when Input = URLs)
# ============================================================
# DOWNLOAD MEDIA  (only runs if INPUT_MODE == "urls")
# Strategy per host:
#   native (YouTube, Vimeo …)       -> yt-dlp direct
#   streamtape.com                  -> pure HTTP (robotlink JS parse)
#   mxdrop.to / mixdrop             -> pure HTTP (Dean-Edwards unpack)
#   voe.sx                          -> pure HTTP (page-source HLS URL)
#   kamehaus / listeamed / lulu.st /
#     turtleviplay / kame.upns.one  -> pure HTTP (DoodStream pass_md5)
#   all others                      -> Playwright intercept (improved)
# ============================================================

if INPUT_MODE != "urls":
    print("Skipping download step (INPUT_MODE != 'urls').")
else:
    if not DOWNLOAD_URLS:
        print("INPUT_MODE is 'urls' but DOWNLOAD_URLS is empty.")
    else:
        import asyncio, nest_asyncio, requests, re, base64, random, string
        nest_asyncio.apply()
        from playwright.async_api import async_playwright
        from urllib.parse import urlparse
        from jsbeautifier.unpackers import packer as jspacker

        UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"
        VIDEO_EXTS = (".mp4", ".m3u8", ".mkv", ".ts")
        SKIP_HOSTS = ("google-analytics", "googleapis", "gstatic",
                      "doubleclick", "facebook", "analytics")
        # These URL patterns are preview/promo clips – filter them in the
        # Playwright fallback so we don't stop at the 15-second teaser.
        PREVIEW_MARKERS = ("promoV", "ds_promo", "/preview/", "/promo/",
                           "_preview", "_promo", "sample.", "trailer.",
                           "view_video.php")   # DoodStream preview endpoint
        PLAY_SELECTORS = [
            '[class*="dooplay"]', '[id*="dooplay"]',   # DoodStream
            ".vjs-big-play-button",                     # Video.js
            ".plyr__control--overlaid",                 # Plyr
            '[class*="jw-icon-display"]',               # JW Player
            '[class*="play-btn"]', '[id*="play-btn"]',
            '[class*="playBtn"]',  '[id*="playBtn"]',
            "video",
        ]
        YTDLP_NATIVE = ("youtube.com", "youtu.be", "vimeo.com",
                        "dailymotion.com", "twitch.tv", "bilibili.com",
                        "twitter.com", "x.com", "tiktok.com")
        # All DoodStream-based embed hosts
        DOOD_HOSTS = ("kamehaus.net", "listeamed.net", "lulu.st",
                      "turtleviplay.xyz", "kame.upns.one")

        def is_native(url):
            h = urlparse(url).netloc.lower()
            return any(h == n or h.endswith("." + n) for n in YTDLP_NATIVE)

        def safe_slug(embed_url, idx):
            """Filesystem-safe filename slug from the embed URL path component."""
            raw = embed_url.rstrip("/").split("/")[-1].lstrip("#")
            raw = raw.split("?")[0]               # strip query string
            slug = re.sub(r'[^\w\-]', '_', raw)  # replace unsafe chars
            return slug or f"video_{idx}"

        # ── Pure HTTP: Streamtape ──────────────────────────────────────────
        def resolve_streamtape(embed_url):
            try:
                r = requests.get(embed_url, headers={"User-Agent": UA}, timeout=15)
                html = r.text
                block_m = re.search(r'robotlink.{0,600}innerHTML', html, re.DOTALL)
                if block_m:
                    ctx = html[max(0, block_m.start() - 300): block_m.end() + 300]
                    m1 = re.search(r"innerHTML\s*=\s*['\"]([^'\"]{10,})['\"]", ctx)
                    m2 = re.search(r"innerHTML\s*\+=\s*['\"]([^'\"]{4,})['\"]", ctx)
                    if m1 and m2:
                        return "https:" + m1.group(1) + m2.group(1)[2:]
                m = re.search(r'(//tapecontent\.net/[^\s"\'<>]+token=[^\s"\'<>]+)', html)
                if m:
                    return "https:" + m.group(1)
            except Exception as e:
                print(f"    streamtape resolver error: {e}")
            return None

        # ── Pure HTTP: Mxdrop ──────────────────────────────────────────────
        def resolve_mxdrop(embed_url):
            try:
                r = requests.get(embed_url,
                                 headers={"User-Agent": UA, "Referer": embed_url},
                                 timeout=15)
                html = r.text
                for packed in re.findall(
                        r'(eval\(function\(p,a,c,k,e,(?:r|d)\).*?\)\))', html, re.DOTALL):
                    if jspacker.detect(packed):
                        unpacked = jspacker.unpack(packed)
                        m = re.search(
                            r'(?:wurl|src|file)\s*[=:]\s*["\']([^"\']+\.mp4[^"\']*)["\']',
                            unpacked)
                        if m:
                            return m.group(1)
            except Exception as e:
                print(f"    mxdrop resolver error: {e}")
            return None

        # ── Pure HTTP: Voe.sx ──────────────────────────────────────────────
        def resolve_voe(embed_url):
            try:
                r = requests.get(embed_url, headers={"User-Agent": UA}, timeout=15)
                html = r.text
                m = re.search(r"['\"]hls['\"]\s*:\s*['\"]([^'\"]+\.m3u8[^'\"]*)['\"]", html)
                if m:
                    return m.group(1)
                m = re.search(r'atob\(["\']([A-Za-z0-9+/=]{20,})["\']', html)
                if m:
                    decoded = base64.b64decode(m.group(1)).decode("utf-8", errors="ignore")
                    m2 = re.search(r'(https?://[^\s"\'<>]+\.m3u8[^\s"\'<>]*)', decoded)
                    if m2:
                        return m2.group(1)
            except Exception as e:
                print(f"    voe resolver error: {e}")
            return None

        # ── Pure HTTP: DoodStream variants ─────────────────────────────────
        def resolve_doodstream(embed_url):
            """
            1. Fetch embed page → extract /pass_md5/HASH/TOKEN?expiry=N from JS.
            2. GET that endpoint (reusing session cookies) → CDN base URL.
            3. Append random suffix + query params → full playable URL.
            This works for kamehaus.net, listeamed.net, lulu.st, turtleviplay.xyz.
            """
            try:
                parts  = urlparse(embed_url)
                origin = f"{parts.scheme}://{parts.netloc}"

                sess = requests.Session()
                sess.headers.update({"User-Agent": UA, "Referer": origin})

                r    = sess.get(embed_url, timeout=20)
                html = r.text

                m = re.search(r"(/pass_md5/[^\s'\"\\)]+)", html)
                if not m:
                    return None

                pass_path = m.group(1)
                pass_url  = origin + pass_path

                r2       = sess.get(pass_url, timeout=20, headers={"Referer": embed_url})
                cdn_base = r2.text.strip()

                if not cdn_base.startswith("http"):
                    return None

                rand = ''.join(random.choices(string.ascii_lowercase + string.digits, k=10))
                qi   = pass_path.find("?")
                qs   = pass_path[qi+1:] if qi != -1 else ""

                return cdn_base + rand + (f"?{qs}" if qs else "")

            except Exception as e:
                print(f"    doodstream resolver error: {e}")
            return None

        # ── Generic Playwright interceptor (fallback) ──────────────────────
        async def intercept_video_url(embed_url, wait_ms=15000):
            found = []
            async with async_playwright() as p:
                browser = await p.chromium.launch(headless=True)
                page    = await browser.new_page()

                def on_request(request):
                    u = request.url
                    if any(ext in u for ext in VIDEO_EXTS):
                        if not any(s in u for s in SKIP_HOSTS):
                            if u not in found:
                                print(f"    caught: {u[:100]}")
                                found.append(u)

                page.on("request", on_request)
                try:
                    await page.goto(embed_url, timeout=40000)
                    await page.wait_for_timeout(3000)
                    for sel in PLAY_SELECTORS:
                        try:
                            await page.click(sel, timeout=1000)
                            print(f"    clicked: {sel}")
                            break
                        except Exception:
                            pass
                    await page.wait_for_timeout(wait_ms)
                except Exception as e:
                    print(f"    page load warning: {e}")
                await browser.close()

            if not found:
                return None
            real = [u for u in found if not any(mk in u for mk in PREVIEW_MARKERS)]
            candidates = real if real else found
            result = candidates[-1]
            print(f"    selected ({len(candidates)}/{len(found)}): {result[:100]}")
            return result

        # ── Downloaders ────────────────────────────────────────────────────
        def dl_ffmpeg(video_url, referer, out_path):
            return subprocess.run([
                "ffmpeg", "-y",
                "-user_agent", UA,
                "-headers", f"Referer: {referer}\r\nOrigin: {referer}\r\n",
                "-i", video_url, "-c", "copy", out_path,
            ]).returncode

        def dl_ytdlp(video_url, referer, out_path, is_hls=False):
            cmd = [
                "yt-dlp", "--no-check-certificate",
                "--referer", referer,
                "--add-header", f"Origin: {referer}",
                "-o", out_path, "--merge-output-format", "mp4",
            ]
            if is_hls:
                cmd += ["--hls-prefer-native"]
            cmd.append(video_url)
            return subprocess.run(cmd).returncode

        # ── Main loop ──────────────────────────────────────────────────────
        Path(INPUT_FOLDER).mkdir(parents=True, exist_ok=True)
        results = {}

        for idx, embed_url in enumerate(DOWNLOAD_URLS, start=1):
            print(f"\n{'='*60}")
            print(f"[{idx}/{len(DOWNLOAD_URLS)}] {embed_url}")

            slug     = safe_slug(embed_url, idx)
            out_path = str(Path(INPUT_FOLDER) / f"{slug}.mp4")
            host     = urlparse(embed_url).netloc.lower()
            video_url = None

            # Branch 1: yt-dlp native
            if is_native(embed_url):
                print("  yt-dlp native mode")
                rc = subprocess.run([
                    "yt-dlp", "--restrict-filenames",
                    "-o", str(Path(INPUT_FOLDER) / "%(title)s.%(ext)s"),
                    embed_url,
                ]).returncode
                results[embed_url] = "done" if rc == 0 else "failed (yt-dlp)"
                print(f"  -> {results[embed_url]}")
                continue

            # Branch 2a: Streamtape pure HTTP
            if "streamtape" in host:
                print("  streamtape HTTP resolver ...")
                video_url = resolve_streamtape(embed_url)
                if video_url:
                    print(f"  resolved: {video_url[:80]}")
                    rc = dl_ffmpeg(video_url, embed_url, out_path)
                    if rc != 0:
                        rc = dl_ytdlp(video_url, embed_url, out_path)
                    results[embed_url] = "done" if rc == 0 else "failed (download)"
                    print(f"  -> {results[embed_url]}")
                    continue
                print("  HTTP resolver failed, falling back to Playwright ...")

            # Branch 2b: Mxdrop pure HTTP
            elif "mxdrop" in host or "mixdrop" in host:
                print("  mxdrop HTTP resolver ...")
                video_url = resolve_mxdrop(embed_url)
                if video_url:
                    print(f"  resolved: {video_url[:80]}")
                    rc = dl_ffmpeg(video_url, embed_url, out_path)
                    if rc != 0:
                        rc = dl_ytdlp(video_url, embed_url, out_path)
                    results[embed_url] = "done" if rc == 0 else "failed (download)"
                    print(f"  -> {results[embed_url]}")
                    continue
                print("  HTTP resolver failed, falling back to Playwright ...")

            # Branch 2c: Voe.sx pure HTTP
            elif "voe.sx" in host:
                print("  voe HTTP resolver ...")
                video_url = resolve_voe(embed_url)
                if video_url:
                    print(f"  resolved: {video_url[:80]}")
                    rc = dl_ytdlp(video_url, embed_url, out_path, is_hls=True)
                    if rc != 0:
                        rc = dl_ffmpeg(video_url, embed_url, out_path)
                    results[embed_url] = "done" if rc == 0 else "failed (download)"
                    print(f"  -> {results[embed_url]}")
                    continue
                print("  HTTP resolver failed, falling back to Playwright ...")

            # Branch 2d: DoodStream variants pure HTTP
            elif any(d in host for d in DOOD_HOSTS):
                print("  doodstream HTTP resolver ...")
                video_url = resolve_doodstream(embed_url)
                if video_url:
                    print(f"  resolved: {video_url[:80]}")
                    rc = dl_ffmpeg(video_url, embed_url, out_path)
                    if rc != 0:
                        rc = dl_ytdlp(video_url, embed_url, out_path)
                    results[embed_url] = "done" if rc == 0 else "failed (download)"
                    print(f"  -> {results[embed_url]}")
                    continue
                print("  HTTP resolver failed, falling back to Playwright ...")

            # Branch 3: Playwright intercept (fallback for all hosts)
            print("  Playwright intercept ...")
            video_url = asyncio.get_event_loop().run_until_complete(
                intercept_video_url(embed_url)
            )

            if not video_url:
                results[embed_url] = "failed (no URL intercepted)"
                print(f"  -> {results[embed_url]}")
                continue

            is_hls = ".m3u8" in video_url

            if "streamtape" in host or "tapecontent" in video_url:
                print("  streamtape CDN -> ffmpeg")
                rc = dl_ffmpeg(video_url, embed_url, out_path)
                if rc != 0:
                    rc = dl_ytdlp(video_url, embed_url, out_path, is_hls)
            elif is_hls:
                print("  HLS -> yt-dlp")
                rc = dl_ytdlp(video_url, embed_url, out_path, is_hls=True)
                if rc != 0:
                    rc = dl_ffmpeg(video_url, embed_url, out_path)
            else:
                print("  direct mp4 -> ffmpeg")
                rc = dl_ffmpeg(video_url, embed_url, out_path)
                if rc != 0:
                    rc = dl_ytdlp(video_url, embed_url, out_path, is_hls)

            results[embed_url] = "done" if rc == 0 else "failed (download)"
            print(f"  -> {results[embed_url]}")

        print("\n\n=== DOWNLOAD SUMMARY ===")
        for url, status in results.items():
            print(f"{status}  {url}")


Skipping download step (INPUT_MODE != 'urls').


In [4]:
# @title ⚙️ Step 5 — Load Whisper model & translation helpers
# ============================================================
# GPU DETECTION + LOAD MODEL
# ============================================================

from faster_whisper import WhisperModel

def gpu_available():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        return result.returncode == 0
    except Exception:
        return False

USE_GPU = gpu_available()
device = "cuda" if USE_GPU else "cpu"
compute_type = COMPUTE_TYPE_GPU if USE_GPU else COMPUTE_TYPE_CPU

print(f"GPU available: {USE_GPU}")
print(f"Loading model: {MODEL_NAME} (device={device}, compute_type={compute_type})")

model = WhisperModel(MODEL_NAME, device=device, compute_type=compute_type)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def format_timestamp(seconds):
    milliseconds = round(seconds * 1000)
    hours = milliseconds // 3600000
    milliseconds %= 3600000
    minutes = milliseconds // 60000
    milliseconds %= 60000
    secs = milliseconds // 1000
    milliseconds %= 1000
    return f"{hours:02}:{minutes:02}:{secs:02},{milliseconds:03}"

def write_srt(segments, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        for idx, seg in enumerate(segments, start=1):
            f.write(f"{idx}\n")
            f.write(f"{format_timestamp(seg['start'])} --> {format_timestamp(seg['end'])}\n")
            f.write(seg["text"].strip())
            f.write("\n\n")

def extract_audio(video_path, output_audio):
    cmd = [
        "ffmpeg", "-y", "-i", str(video_path),
        "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
        str(output_audio),
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def get_duration(path):
    try:
        cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration",
               "-of", "default=noprint_wrappers=1:nokey=1", str(path)]
        result = subprocess.run(cmd, capture_output=True, text=True)
        return float(result.stdout.strip())
    except Exception:
        return 0


# ------------------------------------------------------------
# TRANSLATION HELPERS
#
# Uses Gemini 1.5 Flash (free tier, no credit card required).
# Segments sent in blocks of 20 with 5-line context overlap
# so Gemini can resolve implicit subjects (Japanese omits them).
# Free tier: 1,500 requests/day, 15 RPM -- plenty for subtitles.
# ------------------------------------------------------------

import re   as _re
import requests as _rq
import time as _time

_LANG_NAMES = {
    "ja": "Japanese", "zh": "Chinese", "ko": "Korean",
    "es": "Spanish",  "fr": "French",  "de": "German",
    "pt": "Portuguese", "it": "Italian", "ru": "Russian", "ar": "Arabic",
}

# Free tier: 15 requests per minute
_GEMINI_RPM   = 15
_GEMINI_SLEEP = 60.0 / _GEMINI_RPM   # ~4 s between blocks

_GEMINI_SYSTEM = (
    "You are translating {src} subtitles to {tgt}.\n"
    "Rules:\n"
    "- {src} omits subjects (I / he / she / they). "
    "Resolve them from conversation context (who is talking about whom).\n"
    "- Keep lines short and natural for subtitles.\n"
    "- Preserve proper names, food terms, and exclamations "
    "(e.g. やばい -> 'Oh no'/'That's bad'; うまそう -> 'Looks delicious').\n"
    "- Single-word responses (はい, うん, ほんと, そっか, 納得) "
    "-> brief natural equivalents (Yes / Yeah / Really / I see / Makes sense).\n"
    "- Return ONLY numbered lines in the format [N] translation, no extra text."
)


def translate_segments(segments_list, src_lang, tgt_lang="en",
                        block_size=20, context_size=5, api_key=None):
    """
    Translate subtitle segments with Gemini 1.5 Flash + sliding context.

    Segments processed in blocks; the preceding `context_size` already-
    translated lines are included in each prompt so Gemini can track who
    is talking about whom and assign the correct subject (he/she vs I).

    Parameters
    ----------
    segments_list : list[dict]  dicts must have a 'text' key.
        Modified in-place: each dict receives a 'translated' key.
    src_lang      : str   ISO-639-1 source language code
    tgt_lang      : str   ISO-639-1 target (default 'en')
    block_size    : int   segments per API call (default 20)
    context_size  : int   preceding translated lines as context (default 5)
    api_key       : str | None  Gemini key; falls back to GEMINI_API_KEY

    Returns
    -------
    list[str]  translated texts, same order as segments_list
    """
    if not segments_list:
        return []

    key = api_key or GEMINI_API_KEY
    if not key:
        raise ValueError(
            "No Gemini API key found.\n"
            "Enter it in the Translation settings widget or set GEMINI_API_KEY.\n"
            "Free key (no credit card): https://aistudio.google.com"
        )

    src_name = _LANG_NAMES.get(src_lang, src_lang)
    tgt_name = _LANG_NAMES.get(tgt_lang, tgt_lang)
    system   = _GEMINI_SYSTEM.format(src=src_name, tgt=tgt_name)
    results  = [""] * len(segments_list)
    endpoint = (
        "https://generativelanguage.googleapis.com/v1beta"
        f"/models/gemini-1.5-flash:generateContent?key={key}"
    )

    for i in range(0, len(segments_list), block_size):
        block     = segments_list[i : i + block_size]
        ctx_start = max(0, i - context_size)
        context   = segments_list[ctx_start : i]

        prompt = ""
        if context:
            prompt += (
                "Already-translated context "
                "(do NOT retranslate -- use for subject/speaker tracking):\n"
            )
            for k, s in enumerate(context):
                prompt += f"[{ctx_start + k + 1}] {s.get('translated', s['text'])}\n"
            prompt += "\n"

        prompt += f"Translate these {src_name} subtitle lines to natural {tgt_name}:\n"
        for j, seg in enumerate(block):
            prompt += f"[{i + j + 1}] {seg['text']}\n"

        resp = _rq.post(
            endpoint,
            headers={"Content-Type": "application/json"},
            json={
                "system_instruction": {"parts": [{"text": system}]},
                "contents":           [{"parts": [{"text": prompt}]}],
                "generationConfig":   {"temperature": 0.1, "maxOutputTokens": 1024},
            },
            timeout=60,
        )
        resp.raise_for_status()
        raw = resp.json()["candidates"][0]["content"]["parts"][0]["text"]

        parsed = {}
        for line in raw.strip().splitlines():
            m = _re.match(r"\[(\d+)\]\s*(.*)", line)
            if m:
                parsed[int(m.group(1))] = m.group(2).strip()

        for j, seg in enumerate(block):
            idx               = i + j + 1
            translation       = parsed.get(idx, seg["text"])  # fallback: original
            results[i + j]    = translation
            seg["translated"] = translation  # context for next block

        end_idx = min(i + block_size, len(segments_list))
        print(f"  Translated segments {i + 1}-{end_idx}/{len(segments_list)}")

        if end_idx < len(segments_list):
            _time.sleep(_GEMINI_SLEEP)   # stay within 15 RPM free-tier limit

    return results


def write_srt_translated(segments, output_file):
    """Write SRT using the 'translated' field if present, else 'text'."""
    with open(output_file, "w", encoding="utf-8") as f:
        for idx, seg in enumerate(segments, start=1):
            text = seg.get("translated", seg["text"]).strip()
            f.write(f"{idx}\n")
            f.write(
                f"{format_timestamp(seg['start'])} --> {format_timestamp(seg['end'])}\n"
            )
            f.write(text)
            f.write("\n\n")


GPU available: True
Loading model: large-v3 (device=cuda, compute_type=float16)


In [5]:
# @title ⚙️ Step 6 — Discover files
# ============================================================
# FILE DISCOVERY
# ============================================================

input_root = Path(INPUT_FOLDER)
output_root = Path(OUTPUT_FOLDER)

files = sorted(
    f for f in input_root.rglob("*")
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
)

print(f"Found {len(files)} file(s) to process")
for f in files:
    print(f"  - {f.relative_to(input_root)}")

Found 1 file(s) to process
  - GS-2061 Memorandum play Special Selection Mother and S.mp3


In [6]:
# @title ⚙️ Step 7 — Transcribe / translate
# ============================================================
# PROCESSING
# ============================================================

success = 0
failed = 0
total_duration = 0
overall_start = time.time()

for index, file_path in enumerate(files, start=1):
    file_start = time.time()

    try:
        relative = file_path.relative_to(input_root)
        out_dir = output_root / relative.parent
        out_dir.mkdir(parents=True, exist_ok=True)

        stem = file_path.stem

        # Determine which output paths matter for the skip-check
        srt_path  = out_dir / f"{stem}.srt"
        txt_path  = out_dir / f"{stem}.txt"
        json_path = out_dir / f"{stem}.json"

        if srt_path.exists():
            print(f"[{index}/{len(files)}] SKIPPED (already done): {file_path.name}")
            continue

        print()
        print("=" * 60)
        print(f"[{index}/{len(files)}] Processing: {file_path.name}")

        source_audio = file_path
        temp_audio = None

        if file_path.suffix.lower() in VIDEO_EXTENSIONS:
            temp_audio = Path("/content") / f"temp_{int(time.time())}.wav"
            extract_audio(file_path, temp_audio)
            source_audio = temp_audio

        duration = get_duration(source_audio)
        total_duration += duration

        # Always transcribe first (more accurate than one-shot translate)
        effective_task = "transcribe"

        _LANG_PROMPTS = {
            "es": "La siguiente es una transcripción en español latinoamericano con argot coloquial mexicano.",
            "fr": "Ce qui suit est une transcription en français.",
            "de": "Das Folgende ist eine Transkription auf Deutsch.",
            "ja": "以下は日本語の文字起こしです。",
            "zh": "以下是中文转录。",
            "ko": "다음은 한국어 전사입니다.",
            "pt": "O seguinte é uma transcrição em português.",
            "it": "Di seguito è una trascrizione in italiano.",
            "ru": "Следующее является транскрипцией на русском языке.",
            "ar": "ما يلي نسخ باللغة العربية.",
        }
        _initial_prompt = _LANG_PROMPTS.get(SOURCE_LANGUAGE or "", None)

        segments_generator, info = model.transcribe(
            str(source_audio),
            language=SOURCE_LANGUAGE,
            task=effective_task,
            beam_size=BEAM_SIZE,
            best_of=5,
            temperature=0.0,
            condition_on_previous_text=False,
            compression_ratio_threshold=2.8,
            log_prob_threshold=-1.5,
            no_speech_threshold=0.8,
            vad_filter=True,
            vad_parameters={
                "min_silence_duration_ms": 500,
                "threshold": 0.5,
                "speech_pad_ms": 400,
            },
            word_timestamps=True,
            initial_prompt=_initial_prompt,
        )

        segments = []
        transcript_parts = []

        for seg in segments_generator:
            segments.append({"start": seg.start, "end": seg.end, "text": seg.text})
            transcript_parts.append(seg.text)

        transcript = "\n".join(transcript_parts)

        # Drop hallucinated long-duration segments
        MAX_SEG_DURATION = 10  # seconds
        before = len(segments)
        segments = [s for s in segments if (s["end"] - s["start"]) <= MAX_SEG_DURATION]
        if len(segments) < before:
            print(f"  ⚠️  Dropped {before - len(segments)} hallucinated long segment(s)")

        _detected_lang = info.language
        _src_lang = SOURCE_LANGUAGE or _detected_lang

        # ── Optional: save original-language .txt ──────────────────────────
        if SAVE_ORIG_TXT:
            orig_txt_path = out_dir / f"{stem}.{_src_lang}.txt"
            with open(orig_txt_path, "w", encoding="utf-8") as f:
                f.write(transcript)

        # ── Translation branch ─────────────────────────────────────────────
        if TASK == "translate" and _src_lang != "en":
            print(f"  Translating {len(segments)} segments ({_src_lang}→en) with Gemini 1.5 Flash...")
            translations = translate_segments(segments, _src_lang, "en")
            translated_transcript = "\n".join(translations)

            # SRT uses translated text in translate mode
            write_srt_translated(segments, srt_path)

            # Optional .txt = English translation
            if SAVE_TXT:
                with open(txt_path, "w", encoding="utf-8") as f:
                    f.write(translated_transcript)

            print("  ✅ Transcription + Gemini translation done")
        else:
            # No translation: Whisper task="translate" already gave English,
            # or source is already English.
            for seg in segments:
                seg["translated"] = seg["text"]

            write_srt(segments, srt_path)

            if SAVE_TXT:
                with open(txt_path, "w", encoding="utf-8") as f:
                    f.write(transcript)

            print("  ✅ Transcription saved (original language)")

        # ── Optional: .json metadata ───────────────────────────────────────
        if SAVE_JSON:
            _translation_model_used = (
                "gemini-1.5-flash"
                if TASK == "translate" and _src_lang != "en"
                else "none (no translation)"
            )
            metadata = {
                "file": str(file_path),
                "task": effective_task,
                "translation_model": _translation_model_used,
                "detected_language": info.language,
                "language_probability": info.language_probability,
                "duration_seconds": duration,
                "segments": segments,
            }
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(metadata, f, ensure_ascii=False, indent=2)

        if temp_audio and temp_audio.exists():
            temp_audio.unlink()

        success += 1
        elapsed   = time.time() - overall_start
        avg       = elapsed / index
        remaining = (len(files) - index) * avg

        print(f"Completed in {time.time() - file_start:.1f}s")
        print(f"ETA: {timedelta(seconds=int(remaining))}")

    except Exception as e:
        failed += 1
        print()
        print(f"FAILED: {file_path}")
        print(str(e))



[1/1] Processing: GS-2061 Memorandum play Special Selection Mother and S.mp3
  ⚠️  Dropped 60 hallucinated long segment(s)
  ✅ Transcription saved (original language)
Completed in 400.7s
ETA: 0:00:00


In [ ]:
# @title ⚙️ Step 8 — Print summary
# ============================================================
# SUMMARY
# ============================================================

overall_elapsed = time.time() - overall_start

print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Success: {success}")
print(f"Failed: {failed}")
print(f"Audio duration:  {timedelta(seconds=int(total_duration))}")
print(f"Processing time: {timedelta(seconds=int(overall_elapsed))}")
print()
print(f"Outputs saved to: {OUTPUT_FOLDER}")
print("Finished.")


SUMMARY
Success: 0
Failed: 1
Audio duration:  2:04:51
Processing time: 0:15:21

Outputs saved to: /content/drive/MyDrive/WhisperJAV/output
Finished.


In [ ]:
# @title ⚙️ Step 9 — Download ZIP (only when Output = Session)
# ============================================================
# DOWNLOAD OUTPUTS (only runs if OUTPUT_MODE == "session")
# ============================================================

if OUTPUT_MODE == "session":
    import shutil
    from google.colab import files as colab_files

    zip_base = "/content/whisper_output"
    shutil.make_archive(zip_base, "zip", OUTPUT_FOLDER)
    print(f"Created {zip_base}.zip - starting download...")
    colab_files.download(f"{zip_base}.zip")
else:
    print("OUTPUT_MODE is 'drive' - results are already saved in your Drive at:")
    print(OUTPUT_FOLDER)

OUTPUT_MODE is 'drive' - results are already saved in your Drive at:
/content/drive/MyDrive/WhisperJAV/output


In [ ]:
# @title ⚙️ Step 10 — Auto-disconnect runtime
# ============================================================
# AUTO-DISCONNECT (runs after everything is done)
# ============================================================

print()
print("=" * 70)
print("All done! Disconnecting Colab runtime to save GPU quota...")
print("=" * 70)

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Could not auto-disconnect: {e}")



All done! Disconnecting Colab runtime to save GPU quota...
